Connected to reactive-agent (3.11.x) (Python 3.11.-1)

 # Email sender tool

 **One job:** send email from an async context without blocking the event loop.

 `smtplib` is synchronous — calling it directly inside an `async` function
 blocks the entire event loop until the SMTP handshake finishes.
 `run_in_executor` offloads it to a thread pool, freeing the event loop immediately.

![Email send diagramm](image/email.png)

In [ ]:
import asyncio
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from langchain_core.tools import tool
from app.core.config import get_settings
from app.core.logging import get_logger
import re

log = get_logger(__name__)
settings = get_settings()
_EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

 ## `_send_smtp`

 Synchronous — runs in the thread pool via `run_in_executor`.

 Validates both `to` and `cc` with a regex **before** opening any SMTP connection
 — fail fast, before touching the network.

 **Note:** the docstring in the original is placed after the validation logic,
 which means Python ignores it — a docstring must be the first statement of a function.
 Fixed here by moving it to the top.

In [ ]:
def _send_smtp(to: str, subject: str, body: str, cc: str) -> None:
    """Synchronous SMTP execution — called from run_in_executor."""
    if not _EMAIL_RE.match(to):
        raise ValueError(f"Invalid email address: {to!r}")
    if cc and not _EMAIL_RE.match(cc):
        raise ValueError(f"Invalid CC address: {cc!r}")

    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"]    = settings.smtp_from_email
    msg["To"]      = to
    if cc:
        msg["Cc"] = cc
    msg.attach(MIMEText(body, "plain", "utf-8"))

    with smtplib.SMTP_SSL(settings.smtp_host, settings.smtp_port) as server:
        server.login(settings.smtp_user, settings.smtp_password)
        recipients = [to] + ([cc] if cc else [])
        server.sendmail(settings.smtp_from_email, recipients, msg.as_string())

 ## `send_email`

 The `@tool` decorator exposes this to the LangChain agent.
 The docstring is the description the LLM reads when deciding whether to call it —
 the `IMPORTANT` line nudges the agent to trigger human approval before invoking.
 Enforcement lives in `human_loop.py`, not here.

 **Bug fix:** `asyncio.get_event_loop()` is deprecated in Python 3.10+ inside
 an async function — replaced with `asyncio.get_running_loop()`.

 **Error handling — two branches:**
 - `SMTPAuthenticationError` — points directly at credentials, most common failure
 - Generic `Exception` — returns the exception type without leaking internal details

 Both return strings instead of raising — the agent receives the error as a
 `ToolMessage` and handles it without crashing the graph.

In [ ]:
@tool
async def send_email(to: str, subject: str, body: str, cc: str = "") -> str:
    """
    Send an email.
    IMPORTANT: use this tool only after user confirmation (requires human-in-the-loop).
    """
    try:
        loop = asyncio.get_running_loop()  
        await loop.run_in_executor(None, _send_smtp, to, subject, body, cc)
        log.info("email_sent: to=%s subject=%s", to, subject[:50])
        return f"Email sent successfully to {to} — subject: {subject}"
    except smtplib.SMTPAuthenticationError:
        log.error("email_auth_error: smtp_user=%s", settings.smtp_user)
        return "SMTP authentication error — verify credentials"
    except Exception as e:
        log.error("email_send_error: type=%s error=%s", type(e).__name__, e)
        return f"Error sending email: {type(e).__name__}"